# Temperature Prediction First Data Check

This project will study whether climate indices can help predict next-season temperature anomaly.

This notebook is only a first data check.

The final project will use ERA5 2m temperature, Niño 3.4, PDO, and AO.

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Load the ERA5 temperature file.
era5_data = xr.open_dataset("data/ERA5_2mtemp_1x1.nc")

# Find the temperature variable.
temperature_variable = list(era5_data.data_vars)[0]
era5_temperature = era5_data[temperature_variable]

# Check the basic structure.
print("Temperature variable:", temperature_variable)
print("Dimensions:", dict(era5_temperature.sizes))
print("Time range:", str(era5_temperature.time.min().values)[:10], "to", str(era5_temperature.time.max().values)[:10])
print("Units:", era5_temperature.attrs.get("units", "not listed"))

# Use float64 only for this summary check.
era5_temperature_check = era5_temperature.astype("float64")

# Make a simple describe-style summary.
era5_description = pd.Series({
    "count": int(era5_temperature_check.count().values),
    "mean": float(era5_temperature_check.mean().values),
    "std": float(era5_temperature_check.std().values),
    "min": float(era5_temperature_check.min().values),
    "max": float(era5_temperature_check.max().values),
})

print("Describe:")
print(era5_description)
print("Description: ERA5 has monthly 2m temperature on a global 1x1 grid. The values are in Kelvin.")

In [ ]:
# This helper reads files with one year and 12 monthly values.
def load_monthly_index(file_path, column_name):
    rows = []
    missing_values = [99.99, -99.99, 999, -999, -9999]

    with open(file_path, "r") as data_file:
        for line in data_file:
            parts = line.split()

            # Keep only data rows.
            if len(parts) == 13 and parts[0].isdigit():
                year = int(parts[0])

                # Convert the 12 month columns into one monthly time series.
                for month, value in enumerate(parts[1:], start=1):
                    rows.append({
                        "date": pd.Timestamp(year=year, month=month, day=1),
                        column_name: float(value),
                    })

    data = pd.DataFrame(rows).set_index("date").sort_index()

    # Replace clear missing values with NaN.
    data = data.replace(missing_values, np.nan)

    return data

# Load Niño 3.4 monthly data.
nino34_data = load_monthly_index("data/nina34.anom.data", "nino34")

# Check the data.
print("Head:")
print(nino34_data.head())
print("Tail:")
print(nino34_data.tail())
print("Describe:")
print(nino34_data.describe())
print("Missing value count:")
print(nino34_data.isna().sum())
print("Date range:", nino34_data.index.min(), "to", nino34_data.index.max())
print("Description: Niño 3.4 is a monthly climate index. Some early and future placeholder values are now NaN.")

In [ ]:
# Load PDO monthly data with the same helper.
pdo_data = load_monthly_index("data/ersst.v5.pdo.dat", "pdo")

# Check the data.
print("Head:")
print(pdo_data.head())
print("Tail:")
print(pdo_data.tail())
print("Describe:")
print(pdo_data.describe())
print("Missing value count:")
print(pdo_data.isna().sum())
print("Date range:", pdo_data.index.min(), "to", pdo_data.index.max())
print("Description: PDO is a monthly climate index. Clear placeholder values are now NaN.")

In [ ]:
# Load AO monthly data.
ao_data = pd.read_csv(
    "data/ao.long.csv",
    names=["date", "ao"],
    header=0,
    skipinitialspace=True,
)

# Parse dates and values.
ao_data["date"] = pd.to_datetime(ao_data["date"])
ao_data["ao"] = pd.to_numeric(ao_data["ao"], errors="coerce")
ao_data = ao_data.set_index("date").sort_index()

# Replace clear missing values with NaN.
missing_values = [99.99, -99.99, 999, -999, -9999]
ao_data = ao_data.replace(missing_values, np.nan)

# Check the data.
print("Head:")
print(ao_data.head())
print("Tail:")
print(ao_data.tail())
print("Describe:")
print(ao_data.describe())
print("Missing value count:")
print(ao_data.isna().sum())
print("Date range:", ao_data.index.min(), "to", ao_data.index.max())
print("Description: AO is a monthly climate index. The final placeholder values are now NaN.")

In [ ]:
# Combine the three climate indices.
climate_indices = pd.concat([nino34_data, pdo_data, ao_data], axis=1).sort_index()

# Drop rows where all three values are missing.
climate_indices = climate_indices.dropna(how="all")

# Check the combined data.
print("Head:")
print(climate_indices.head())
print("Tail:")
print(climate_indices.tail())
print("Describe:")
print(climate_indices.describe())
print("Missing value count:")
print(climate_indices.isna().sum())
print("Date range:", climate_indices.index.min(), "to", climate_indices.index.max())
print("Correlation matrix:")
print(climate_indices.corr())
print("Description: The combined table keeps all available months. Missing values mostly come from different start and end dates.")

In [ ]:
# Plot the three climate indices.
climate_indices.plot(figsize=(10, 4))
plt.title("Climate Indices Monthly Time Series")
plt.xlabel("Date")
plt.ylabel("Index value")
plt.tight_layout()
plt.show()

## Simple Next Plan

- Choose one location from ERA5, such as Boulder, Colorado.
- Convert monthly temperature to seasonal mean.
- Compute temperature anomaly.
- Convert climate indices to seasonal mean.
- Use current-season indices to predict next-season temperature anomaly.
- Start with simple multiple linear regression.
- Later add out-of-sample validation and residual checks.